# Live Tomo Diff Notebook

This notebook wraps `live_tomo_diff.py` with widgets so you can:

- browse a collection directory
- choose the reference and comparison datasets
- pick a projection index across all detector block files
- choose whether live mode should follow only the same position or all positions
- preview a diff image inside the notebook
- launch the live viewer with the selected options

If you leave the comparison dataset as `AUTO_FOLLOW`, the launched viewer will watch for the newest matching dataset in the collection.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

import live_tomo_diff as ltd

plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
AUTO_FOLLOW = 'AUTO_FOLLOW'

collection_dir = widgets.Text(
    value='011_test',
    description='Collection:',
    layout=widgets.Layout(width='700px'),
)
reference_dataset = widgets.Dropdown(
    options=[],
    description='Reference:',
    layout=widgets.Layout(width='700px'),
)
comparison_dataset = widgets.Dropdown(
    options=[AUTO_FOLLOW],
    value=AUTO_FOLLOW,
    description='Comparison:',
    layout=widgets.Layout(width='700px'),
)
projection_index = widgets.BoundedIntText(
    value=0,
    min=0,
    description='Projection:',
    layout=widgets.Layout(width='250px'),
)
position_mode = widgets.Dropdown(
    options=['same', 'all'],
    value='same',
    description='Positions:',
    layout=widgets.Layout(width='250px'),
)
dataset_path = widgets.Text(
    value='',
    placeholder='auto-detect image stack',
    description='Dataset path:',
    layout=widgets.Layout(width='700px'),
)
poll_interval = widgets.FloatSlider(
    value=2.0,
    min=0.2,
    max=10.0,
    step=0.2,
    description='Poll (s):',
    readout_format='.1f',
    layout=widgets.Layout(width='350px'),
)
refresh_button = widgets.Button(description='Refresh datasets', button_style='info')
preview_button = widgets.Button(description='Preview diff', button_style='success')
command_button = widgets.Button(description='Show command')
launch_button = widgets.Button(description='Launch live viewer', button_style='warning')
output = widgets.Output()


def dataset_choices(collection_path: Path):
    if not collection_path.exists():
        return []
    return [
        path.name
        for path in sorted(collection_path.iterdir())
        if ltd.is_dataset_directory(path)
    ]


def refresh_datasets(*_):
    with output:
        clear_output(wait=True)
        collection_path = Path(collection_dir.value).expanduser()
        choices = dataset_choices(collection_path)
        if not choices:
            print(f'No dataset directories found in {collection_path}')
            reference_dataset.options = []
            comparison_dataset.options = [AUTO_FOLLOW]
            comparison_dataset.value = AUTO_FOLLOW
            return

        reference_dataset.options = choices
        if reference_dataset.value not in choices:
            reference_dataset.value = choices[0]

        comparison_options = [AUTO_FOLLOW] + choices
        comparison_dataset.options = comparison_options
        if comparison_dataset.value not in comparison_options:
            comparison_dataset.value = AUTO_FOLLOW

        print(f'Loaded {len(choices)} dataset directories from {collection_path}')


def selected_paths():
    collection_path = Path(collection_dir.value).expanduser().resolve()
    reference_path = collection_path / reference_dataset.value
    comparison_value = comparison_dataset.value
    comparison_path = None if comparison_value == AUTO_FOLLOW else collection_path / comparison_value
    explicit_dataset_path = dataset_path.value.strip() or None
    return collection_path, reference_path, comparison_path, explicit_dataset_path


def preview_diff(*_):
    with output:
        clear_output(wait=True)
        try:
            _, reference_path, comparison_path, explicit_dataset_path = selected_paths()
            reference_root, reference_scan = ltd.resolve_input_target(reference_path)
            comparison_root, comparison_scan, auto_follow = ltd.resolve_second_target(
                reference_root,
                comparison_path,
                position_mode.value,
            )
            ref_count = ltd.scan_projection_count(reference_scan, explicit_dataset_path)
            cmp_count = ltd.scan_projection_count(comparison_scan, explicit_dataset_path)
            idx = projection_index.value
            ref_image = ltd.load_projection_radiogram(reference_scan, idx, explicit_dataset_path)
            cmp_image = ltd.load_projection_radiogram(comparison_scan, idx, explicit_dataset_path)
            diff = cmp_image - ref_image

            print('Reference dataset:', reference_root)
            print('Comparison dataset:', comparison_root)
            print('Reference scan:', reference_scan)
            print('Comparison scan:', comparison_scan)
            print('Auto follow:', auto_follow)
            print('Projection index:', idx)
            print('Reference projections:', ref_count)
            print('Comparison projections:', cmp_count)
            print('Diff min/max/mean:', float(diff.min()), float(diff.max()), float(diff.mean()))

            fig, ax = plt.subplots()
            image = ax.imshow(diff, cmap='gray')
            ax.set_title(f'{comparison_root.name} - {reference_root.name}')
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            plt.colorbar(image, ax=ax, label='Difference')
            plt.show()
        except Exception as exc:
            print(f'Preview failed: {exc}')


def build_command():
    _, reference_path, comparison_path, explicit_dataset_path = selected_paths()
    command = [sys.executable, 'live_tomo_diff.py', '--reference-path', str(reference_path)]
    if comparison_path is not None:
        command.extend(['--comparison-path', str(comparison_path)])
    command.extend(['--projection-index', str(projection_index.value)])
    command.extend(['--position-mode', position_mode.value])
    command.extend(['--poll-interval', str(poll_interval.value)])
    if explicit_dataset_path is not None:
        command.extend(['--dataset-path', explicit_dataset_path])
    return command


def show_command(*_):
    with output:
        clear_output(wait=True)
        print(' '.join(shlex.quote(part) for part in build_command()))


def launch_live_viewer(*_):
    with output:
        clear_output(wait=True)
        command = build_command()
        process = subprocess.Popen(command, cwd=str(Path.cwd()))
        print('Launched live viewer PID:', process.pid)
        print('Command:')
        print(' '.join(shlex.quote(part) for part in command))
        print('Close the matplotlib window or stop the process manually when finished.')


refresh_button.on_click(refresh_datasets)
preview_button.on_click(preview_diff)
command_button.on_click(show_command)
launch_button.on_click(launch_live_viewer)

controls_top = widgets.VBox([
    collection_dir,
    reference_dataset,
    comparison_dataset,
    dataset_path,
])
controls_mid = widgets.HBox([
    projection_index,
    position_mode,
    poll_interval,
])
controls_buttons = widgets.HBox([
    refresh_button,
    preview_button,
    command_button,
    launch_button,
])

display(controls_top, controls_mid, controls_buttons, output)
refresh_datasets()

In [ ]:
gallery_indices = widgets.Text(
    value='0, 100, 237, 500',
    description='Gallery:',
    layout=widgets.Layout(width='500px'),
)
gallery_button = widgets.Button(description='Preview gallery', button_style='success')
gallery_output = widgets.Output()


def parse_gallery_indices(raw: str):
    indices = []
    for part in raw.split(','):
        part = part.strip()
        if not part:
            continue
        indices.append(int(part))
    if not indices:
        raise ValueError('Enter at least one projection index.')
    return indices


def preview_gallery(*_):
    with gallery_output:
        clear_output(wait=True)
        try:
            _, reference_path, comparison_path, explicit_dataset_path = selected_paths()
            reference_root, reference_scan = ltd.resolve_input_target(reference_path)
            comparison_root, comparison_scan, auto_follow = ltd.resolve_second_target(
                reference_root,
                comparison_path,
                position_mode.value,
            )
            indices = parse_gallery_indices(gallery_indices.value)
            cols = min(3, len(indices))
            rows = (len(indices) + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
            axes = np.atleast_1d(axes).ravel()

            print('Reference dataset:', reference_root)
            print('Comparison dataset:', comparison_root)
            print('Auto follow:', auto_follow)
            print('Gallery indices:', indices)

            for ax, idx in zip(axes, indices):
                ref_image = ltd.load_projection_radiogram(reference_scan, idx, explicit_dataset_path)
                cmp_image = ltd.load_projection_radiogram(comparison_scan, idx, explicit_dataset_path)
                diff = cmp_image - ref_image
                image = ax.imshow(diff, cmap='gray')
                ax.set_title(f'Projection {idx}')
                ax.set_xlabel('X')
                ax.set_ylabel('Y')
                plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

            for ax in axes[len(indices):]:
                ax.axis('off')

            fig.suptitle(f'{comparison_root.name} - {reference_root.name}')
            fig.tight_layout()
            plt.show()
        except Exception as exc:
            print(f'Gallery preview failed: {exc}')


gallery_button.on_click(preview_gallery)
display(widgets.HBox([gallery_indices, gallery_button]), gallery_output)

## Notes

- `Projection` is zero-based across the whole projection scan, not per `*.h5` block.
- If `Comparison` is `AUTO_FOLLOW`, the notebook preview resolves the current latest matching dataset once, while `Launch live viewer` starts the real polling viewer.
- `Positions = same` keeps auto-follow on the same position label as the reference dataset, such as `first_position`.
- `Positions = all` allows auto-follow across all positions in the collection.